[[이유한님] 캐글 코리아 캐글 스터디 커널 커리큘럼](https://kaggle-kr.tistory.com/32)

[TensorFlow Speech Recognition Challenge](https://www.kaggle.com/c/tensorflow-speech-recognition-challenge)

[WavCeption V1: a 1-D Inception approach (LB 0.76)](https://www.kaggle.com/code/ivallesp/wavception-v1-a-1-d-inception-approach-lb-0-76)

# WavCeption V1: just a 1-D Inception approach

I just wanted to share a little toy I have been playing with and gave me **amazing results**. As I currently don't have time, I would like to share it to see how people plays with it :-D. **The WavCeption V1** network seems to produce impressive results compared to a regular convolutional neural network, but in this competition it seems that there is a hard-work on the pre-processing and unknown tracks management. It is based on the Google's inception network, the same idea.

I wrote some weeks ago a module implementing it so that it is easy to build an 1D-inception network by connecting lots of these modules in cascade (as you will see below).

Unfortunately and due to several Kaggle constraints, it won't run in the kernel machine, so I encourage you to download it and run it in your own machine.

By running the model for 12h without struggling too much I achieved 0.76 in the leaderboard (with 0.84 in local test). Some other trials in the same line gave me 0.89 in local, so there is a huge improvement in how you deal with the unknown clips :-D

# DeepL 번역
제가 가지고 놀다가 **놀라운 결과**를 얻은 작은 장난감을 공유하고 싶었습니다. 지금은 시간이 없어서 사람들이 어떻게 가지고 노는지 보기 위해 공유하고 싶습니다 :-D. **WavCeption V1** 네트워크는 일반 컨볼루션 신경망에 비해 인상적인 결과를 내는 것 같은데, 이번 대회에서는 전처리와 미지의 트랙 관리에 공을 들인 것 같습니다. 구글의 인셉션 네트워크와 같은 아이디어를 기반으로 하고 있습니다.

저는 몇 주 전에 이를 구현하는 모듈을 작성했는데, 아래에서 보시는 것처럼 이 모듈을 많이 계단식으로 연결하면 1D-인셉션 네트워크를 쉽게 구축할 수 있습니다.

안타깝게도 몇 가지 Kaggle 제약 조건으로 인해 커널 머신에서는 실행되지 않으므로 직접 다운로드하여 자신의 머신에서 실행해 보시기 바랍니다.

큰 어려움 없이 12시간 동안 모델을 실행하여 리더보드에서 0.76을 달성했습니다(로컬 테스트에서는 0.84). 같은 라인의 다른 테스트에서는 로컬에서 0.89를 기록했으니, 알 수 없는 클립을 처리하는 방법이 크게 개선되었습니다 :-D

## Load modules and libraries

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
# %matplotlib inline
get_ipython().run_line_magic('matplotlib', 'inline')    # 매직명렁어 안 될 때
import numpy as np
import pandas as pd
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import shutil
import glob
import random
from tqdm import tqdm
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from numpy.fft import irfft
from tqdm import tqdm
from scipy.io import wavfile
import tensorflow as tf

## Noise generation functions

The code in this section has been borrowed and adapted from:  
https://github.com/python-acoustics/python-acoustics/blob/master/acoustics/generator.py

In [ ]:
def ms(x):
    """Mean value of signal 'x' squared.
    :param x: Dynamic quantity.
    :returns: Mean squared of 'x'.
    """
    return (np.abs(x)**2.0).mean()

def normalize(y, x=None):
    """normalize power in y to a (standard normal) white noise signal.
    Optionally normalize to power in signal `x`.
    # The mean power of a Gaussian with :math: `\\mu=0` and :math:`\\sigma=1` is 1.
    """
    # return y * np.sqrt( (np.abs(x)**2.0).mean() / (np.abs(y)**2.0).mean() )
    if x is not None:
        x = ms(x)
    else:
        x = 1.0
    return y * np.sqrt( x / ms(y) )
    # return  y * np.sqrt( 1.0 / np.abs(y)**2.0).mean() )

def white_noise(N, state=None):
    state = np.random.RandomState() if state is None else state
    return state.randn(N)

def pink_noise(N, state=None):
    state = np.random.RandomState() if state is None else state
    uneven = N % 2
    X = state.randn(N // 2 + 1 + uneven) + 1j * state.randn(N // 2 + 1 + uneven)
    S = np.sqrt(np.arange(len(X)) + 1.) # `+ 1` to avoid divide by zero
    y = (irfft(X / S)).real
    if uneven:
        y = y[:-1]
    return normalize(y)

def blue_noise(N, state=None):
    """
    Blue noise
    
    :param N: Amount of samples
    :param state: State of PRNG.
    :type state: :class: `np.random.RandomState`

    Power increases with 6 dB per octave.
    Power density increases with 3 dB per octave.

    """
    state = np.random.RandomState() if state is None else state
    uneven = N % 2
    X = state.randn(N // 2 + 1 + uneven) + 1j * state.randn(N // 2 + 1 + uneven)
    S = np.sqrt(np.arange(len(X)))  # Filter
    y = (irfft(X * S)).real
    if uneven:
        y = y[:-1]
    return normalize(y)


def brown_noise(N, state=None):
    # 함수와 독스의 이름이 다름. 내용이 다음 함수와 다른 걸로 봐선 이름만 잘못된 듯
    """
    Violet noise
    
    :param N: Amount of samples
    :param state: State of PRNG.
    :type state: :class: `np.random.RandomState`

    Power decreases with -3 dB per octave.
    Power density decreases with 6 dB per octave.

    """
    state = np.random.RandomState() if state is None else state
    uneven = N % 2
    X = state.randn(N // 2 + 1 + uneven) + 1j * state.randn(N // 2 + 1 + uneven)
    S = np.sqrt(np.arange(len(X)) + 1)  # Filter
    y = (irfft(X / S)).real
    if uneven:
        y = y[:-1]
    return normalize(y)

def violet_noise(N, state=None):
    """
    Violet noise. Power increases with 6 dB per octave.
    
    :param N: Amount of samples
    :param state: State of PRNG.
    :type state: :class: `np.random.RandomState`

    Power iecreases with +9 dB per octave.
    Power density increases with +6 dB per octave.

    """
    state = np.random.RandomState() if state is None else state
    uneven = N % 2
    X = state.randn(N // 2 + 1 + uneven) + 1j * state.randn(N // 2 + 1 + uneven)
    S = np.arange(len(X)) + 1 # Filter
    y = (irfft(X * S)).real
    if uneven:
        y = y[:-1]
    return normalize(y)


## Tensorflow utilities

Utilities to modularize tensorflow common actions

In [ ]:
tf.config.list_physical_devices("GPU")

In [ ]:
# Tf Utils
## tf.Session은 제거되고 runtime 설정 함수로 역할 변경됨
## tf.ConfigProto -> tf.config.*

def get_tensorflow_configuration(device="0", memory_fraction=1):
    """
    Function for selecting the GPU to use and the amount of memory the process is allowed to use
    :param device: which device should be used (str)
    :param memory_fraction: which proportion of memory must be allocated (float)
    :return: config to be passed to the session (tf object)
    """

    gpus = tf.config.list_physical_devices("GPU")
    
    if not gpus:
        print("No GPU found. Running on CPU")
        return
    
    device = int(device)

    try:
        # 특정 GPU 설정
        tf.config.set_visible_devices(gpus[device], "GPU")

        # 메모리 설정
        if memory_fraction < 1.0:
            total_memory = tf.config.experimental.get_device_details()

            if total_memory:
                tf.config.set_logical_device_configuration(
                    gpus[device],
                    [tf.config.LogicalDeviceConfiguration(
                        memory_limit=total_memory * memory_fraction / (1024**2)
                    )]
                )
        else:
            tf.config.experimental.set_memory_growth(gpus[device], True)
    except RuntimeError as e:
        print(e)

def start_tensorflow_session(device="0", memory_fraction=1.0):
    """
    TF2에서는 tf.Session이 없으므로 초기 설정만 수행
    """
    get_tensorflow_configuration(device, memory_fraction)

# def get_summary_writer(session, logs_path, project_id, version_id):
def get_summary_writer(logs_path, project_id, version_id):
    """
    For Tensorboard reporting
    :param session: opened tensorflow session (tf.Session)
    :param logs_path: path where tensorboard is looking for logs (str)
    :param project_id: name of the project for reporting purpose (str)
    :param version_id: name of the version for reporting purpose (str)
    :return summary_writer: the tensorboard writer
    """
    path = os.path.join(logs_path, "{}_{}".format(project_id, version_id))
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
    summary_writer = tf.summary.create_file_writer(path)
    # summary_writer = tf.summary.create_file_writer(path, graph_def=session.graph_def)
    
    return(summary_writer)

## Paths management module
Modules to deal with the paths

In [ ]:
# Common paths
def _norm_path(path):
    """
    Decorator function intended for using it to normalize a the output of a path retrieval function. Useful for
    fixing the slash/backslash windows cases.
    """
    def normalize_path(*arg, **kwargs):
        return os.path.normpath(path(*arg, **kwargs))
    return normalize_path

def _assure_path_exists(path):
    """
    Decorator function intended for checking the existence of a the output of a path retrieva; function. Useful for
    fixing the slash/backslash windows cases
    """
    def assure_exists(*args, **kwargs):
        p = path(*args, **kwargs)
        assert os.path.exists(p), "the following path does not exist: '{}'".format(p)
        return p
    return assure_exists

def _is_output_path(path):
    """
    Decorator function intended for grouping the functions which are applied over the output of an output path retrieval
    function
    """
    @_norm_path
    @_assure_path_exists
    def check_existence_or_crate_it(*args, **kwargs):
        if not os.path.exists(path(*args, **kwargs)):
            "Path does not exist... creating it: {}".format(path(*args, **kwargs))
            os.makedirs(path(*args, **kwargs))
        return path(*args, **kwargs)
    return check_existence_or_crate_it

def _is_input_path(path):
    """
    Decorator function intended for grouping the functions which are applied over the output of an input path retrieval
    function
    """
    @_norm_path
    @_assure_path_exists
    def check_existence(*args, **kwargs):
        return path(*args, **kwargs)
    return check_existence

@_is_input_path
def get_train_path():
    path = "./input/006_tensorflow-speech-recognition-challenge/train"
    return path

@_is_input_path
def get_test_path():
    path = "./input/006_tensorflow-speech-recognition-challenge/test"
    return path

@_is_input_path
def get_train_audio_path():
    path = os.path.join(get_train_path(), "audio")
    return path

@_is_input_path
def get_scoring_audio_path():
    path = os.path.join(get_test_path(), "audio")
    return path

@_is_output_path
def get_submission_path():
    path = "./working/output"
    return path

@_is_output_path
def get_silence_path():
    path = "./working/silence"
    return path

## Utilities
Common general-purpose utilities

In [ ]:
# Utilities
flatten = lambda l: [item for sublist in l for item in sublist]

def batching(iterable, n = 1):
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx: min(ndx + n, l)]

## Data Tools

Data handling tools

In [ ]:
# Data tools
def read_wav(filepath, pad=True):
    """
    Given the filepath of a wav file, this function reads it, normalizes it and pads
    it to assure it has 16ks samples.
    :param filepath: existing filepath of a wav file (str)
    :param pad: is padding required? (bool)
    :returns: the sample and the target variable (tuple of (np.array, str))
    """
    saomple_rate, x = wavfile.read(filepath)
    target = os.path.split(os.path.split(filepath)[0])[1]
    assert saomple_rate == 16000
    if pad:
        return np.pad(x, (0, 16000 - len(x)), mode="constant") / 32768, target
    else:
        return x/32768, target

def get_batcher(list_of_paths, batch_size, label_encoder=None, scoring=False):
    """
    Builds a batch generator given a list of batches
    :param list_of_paths: list of tuples with elements of format (filepath, target) (list)
    :param batch_size: size of the batch (int)
    :param label_encoder: fitted LabelEncoder (sklearn.LabelEnocoder | optional)
    :returns: batch generator
    """
    for filepaths in batching(list_of_paths, batch_size):
        wavs, targets = zip(*list(map(read_wav, filepaths)))
        if scoring:
            yield np.expand_dims(np.row_stack(wavs), 2), filepaths
        else:
            if label_encoder is None:
                yield np.expand_dims(np.row_stack(wavs), 2), np.row_stack(targets)
            else:
                yield np.expand_dims(np.row_stack(wavs), 2), np.expand_dims(label_encoder.transform(np.squeeze(targets)), 1)

## Architecture building blocks

Inception-1D (a.k.a wavception) is a module I designed some weeks ago for this problem. It substantially enhances the performance of a regular convolutional neural net.

In [ ]:
# class BatchNorm(tf.keras.layers.Layer):
#    #  def __init__(self, epsilon=1e-5, momentum=0.999, name=None):
#     def __init__(self, epsilon=1e-5, momentum=0.999, name="batch_norm"):
#       super().__init__(name=name)
#       self.bn = tf.keras.layers.BatchNormalization(
#          epsilon=epsilon,
#          momentum=momentum
#       )
#       # self.epsilon = epsilon
#       # self.momentum = momentum
#       # self.name = name
        
#     def __call__(self, x, is_train=True):
#         return tf.keras.layers.BatchNormalization(
#                                             x,
#                                             momentum=self.momentum,
#                                             epsilon=self.epsilon,
#                                             scale=True,
#                                             name=self.name,
#                                             training=is_train)
    
# def inception_1d(x, is_train, depth, norm_function, active_function, name=None):
#     """Inception 1D module implementation.
#     :param x: input to current module (4D tensor with channels-last)
#     :param is_train: it is intented to be a boolean placeholder for controling the BatchNormalization behavior (0D tensor)
#     :param depth: linearly controls the depth of the network (int)
#     :param norm_function: normalization class (same format as the BatchNorm class above)
#     :param active_function: tensorflow activation function (e.g. tf.nn.relu)
#     :param name: of the variable scpoe (str)
#     """

#     def __init__(self, depth, norm_function, active_function, name=None):
#         super().__init__(name=name)

#         self.depth = depth
#         self.norm = norm_function
#         self.active = active_function

#         init = tf.keras.initializers.GlorotUniform()
#         x_norm = norm_function(name="norm_input")(x, train=is_train)

#         # Branch 1: 64 x copy 1x1
#         branch_conv_1_1 = tf.keras.layers.Conv1D(filters=16 * depth, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#         branch_conv_1_1 = norm_function(name="norm_conv_1_1")(branch_conv_1_1, train=is_train)
#         branch_conv_1_1 = active_function(branch_conv_1_1, name="activation_1_1")(branch_conv_1_1)
        
#         # Branch 2: 128 x conv 3x3
#         branch_conv_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#         branch_conv_3_3 = norm_function(name="norm_conv_3_3_1")(branch_conv_3_3, train=is_train)
#         branch_conv_3_3 = active_function(branch_conv_3_3, "activation_3_3_1")(branch_conv_3_3)

#         branch_conv_3_3 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=3, padding="same", kernel_initializer=init)(branch_conv_3_3)
#         branch_conv_3_3 = norm_function(name="norm_conv_3_3_2")(branch_conv_3_3, train=is_train)
#         branch_conv_3_3 = active_function(branch_conv_3_3, "activation_3_3_2")(branch_conv_3_3)

#         # Branch 3: 128 x conv 5x5
#         branch_conv_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#         branch_conv_5_5 = norm_function(name="norm_conv_5_5_1")(branch_conv_5_5, train=is_train)
#         branch_conv_5_5 = active_function(branch_conv_5_5, "activation_5_5_1")(branch_conv_5_5)

#         branch_conv_5_5 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=5, padding="same", kernel_initializer=init)(branch_conv_5_5)
#         branch_conv_5_5 = norm_function(name="norm_conv_5_5_2")(branch_conv_5_5, train=is_train)
#         branch_conv_5_5 = active_function(branch_conv_5_5, "activation_5_5_2")(branch_conv_5_5)

#         # Branch 4: 128 x conv 7x7
#         branch_conv_7_7 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#         branch_conv_7_7 = norm_function(name="norm_conv_7_7_1")(branch_conv_7_7, train=is_train)
#         branch_conv_7_7 = active_function(branch_conv_7_7, "activation_7_7_1")(branch_conv_7_7)

#         branch_conv_7_7 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=5, padding="same", kernel_initializer=init)(branch_conv_7_7)
#         branch_conv_7_7 = norm_function(name="norm_conv_7_7_2")(branch_conv_7_7, train=is_train)
#         branch_conv_7_7 = active_function(branch_conv_7_7, "activation_7_7_2")(branch_conv_7_7)

        
#         # Branch 5: 16 x (max_pool 3x3 + conv 1x1)
#         branch_maxpool_3_3 = tf.keras.layers.MaxPooling1D(pool_size=3, strides=1, padding="same", name="maxpool_3")(x_norm)
#         branch_maxpool_3_3 = norm_function(name="norm_maxpool_3_3")(branch_maxpool_3_3, train=is_train)
#         branch_maxpool_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_maxpool_3_3)
#         # Branch 6: 16 x (max_pool 5x5 + conv 1x1)
#         branch_maxpool_5_5 = tf.keras.layers.MaxPooling1D(pool_size=5, strides=1, padding="same", name="maxpool_5")(x_norm)
#         branch_maxpool_5_5 = norm_function(name="norm_maxpool_5_5")(branch_maxpool_5_5, train=is_train)
#         branch_maxpool_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_maxpool_5_5)
#         # Branch 7: 16 x (max_pool 3x3 + conv 1x1)
#         branch_avgpool_3_3 = tf.keras.layers.AveragePooling1D(pool_size=3, strides=1, padding="same", name="avgpool_3")(x_norm)
#         branch_avgpool_3_3 = norm_function(name="norm_avgpool_3_3")(branch_avgpool_3_3, train=is_train)
#         branch_avgpool_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_avgpool_3_3)
#         # Branch 8: 16 x (max_pool 5x5 + conv 1x1)
#         branch_avgpool_5_5 = tf.keras.layers.AveragePooling1D(pool_size=5, strides=1, padding="same", name="avgpool_5")(x_norm)
#         branch_avgpool_5_5 = norm_function(name="norm_avgpool_5_5")(branch_avgpool_5_5, train=is_train)
#         branch_avgpool_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_avgpool_5_5)
      
        
#         # Concatenate
#         output = tf.concat([branch_conv_1_1, branch_conv_3_3, branch_conv_5_5, branch_conv_7_7, branch_maxpool_3_3,
#                             branch_maxpool_5_5, branch_avgpool_3_3, branch_avgpool_5_5], axis=1)
#         return output

In [ ]:
class BatchNorm(tf.keras.layers.Layer):
   #  def __init__(self, epsilon=1e-5, momentum=0.999, name=None):
    def __init__(self, epsilon=1e-5, momentum=0.999, name="batch_norm"):
      super().__init__(name=name)
      self.bn = tf.keras.layers.BatchNormalization(
         epsilon=epsilon,
         momentum=momentum,
         scale=True,
         name=self.name,
      )
    def call(self, x, is_train=True):
         return self.bn(x, training=is_train)
    
class Inception1D(tf.keras.layers.Layer):

    def __init__(self,
                 depth,
                 norm_function=BatchNorm,
                 active_function=tf.nn.relu,
                 name=None):

        super().__init__(name=name)

        self.depth = depth
        self.active = active_function

        init = tf.keras.initializers.GlorotUniform()

        # Input norm
        self.norm_input = norm_function(name="norm_input")

        # Branch 1
        self.conv1_1 = tf.keras.layers.Conv1D(
            filters=16 * depth,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.norm1_1 = norm_function(name="norm_conv_1_1")

        # Branch 2
        self.conv3_3_1 = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.norm3_3_1 = norm_function(name="norm_conv_3_3_1")

        self.conv3_3_2 = tf.keras.layers.Conv1D(
            filters=32 * depth,
            kernel_size=3,
            padding="same",
            kernel_initializer=init
        )

        self.norm3_3_2 = norm_function(name="norm_conv_3_3_2")

        # Branch 3
        self.conv5_5_1 = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.norm5_5_1 = norm_function(name="norm_conv_5_5_1")

        self.conv5_5_2 = tf.keras.layers.Conv1D(
            filters=32 * depth,
            kernel_size=5,
            padding="same",
            kernel_initializer=init
        )

        self.norm5_5_2 = norm_function(name="norm_conv_5_5_2")

        # Branch 4
        self.conv7_7_1 = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.norm7_7_1 = norm_function(name="norm_conv_7_7_1")

        self.conv7_7_2 = tf.keras.layers.Conv1D(
            filters=32 * depth,
            kernel_size=7,
            padding="same",
            kernel_initializer=init
        )

        self.norm7_7_2 = norm_function(name="norm_conv_7_7_2")

        # Pool branches
        self.maxpool3 = tf.keras.layers.MaxPooling1D(
            pool_size=3,
            strides=1,
            padding="same"
        )

        self.norm_max3 = norm_function(name="norm_maxpool_3_3")

        self.maxpool3_conv = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.maxpool5 = tf.keras.layers.MaxPooling1D(
            pool_size=5,
            strides=1,
            padding="same"
        )

        self.norm_max5 = norm_function(name="norm_maxpool_5_5")

        self.maxpool5_conv = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.avgpool3 = tf.keras.layers.AveragePooling1D(
            pool_size=3,
            strides=1,
            padding="same"
        )

        self.norm_avg3 = norm_function(name="norm_avgpool_3_3")

        self.avgpool3_conv = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

        self.avgpool5 = tf.keras.layers.AveragePooling1D(
            pool_size=5,
            strides=1,
            padding="same"
        )

        self.norm_avg5 = norm_function(name="norm_avgpool_5_5")

        self.avgpool5_conv = tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=1,
            padding="same",
            kernel_initializer=init
        )

    def call(self, x, training=False):

        x_norm = self.norm_input(x, training=training)

        # Branch 1
        b1 = self.conv1_1(x_norm)
        b1 = self.norm1_1(b1, training=training)
        b1 = self.active(b1)

        # Branch 2
        b2 = self.conv3_3_1(x_norm)
        b2 = self.norm3_3_1(b2, training=training)
        b2 = self.active(b2)

        b2 = self.conv3_3_2(b2)
        b2 = self.norm3_3_2(b2, training=training)
        b2 = self.active(b2)

        # Branch 3
        b3 = self.conv5_5_1(x_norm)
        b3 = self.norm5_5_1(b3, training=training)
        b3 = self.active(b3)

        b3 = self.conv5_5_2(b3)
        b3 = self.norm5_5_2(b3, training=training)
        b3 = self.active(b3)

        # Branch 4
        b4 = self.conv7_7_1(x_norm)
        b4 = self.norm7_7_1(b4, training=training)
        b4 = self.active(b4)

        b4 = self.conv7_7_2(b4)
        b4 = self.norm7_7_2(b4, training=training)
        b4 = self.active(b4)

        # Branch 5
        b5 = self.maxpool3(x_norm)
        b5 = self.norm_max3(b5, training=training)
        b5 = self.maxpool3_conv(b5)

        # Branch 6
        b6 = self.maxpool5(x_norm)
        b6 = self.norm_max5(b6, training=training)
        b6 = self.maxpool5_conv(b6)

        # Branch 7
        b7 = self.avgpool3(x_norm)
        b7 = self.norm_avg3(b7, training=training)
        b7 = self.avgpool3_conv(b7)

        # Branch 8
        b8 = self.avgpool5(x_norm)
        b8 = self.norm_avg5(b8, training=training)
        b8 = self.avgpool5_conv(b8)

        output = tf.concat(
            [b1, b2, b3, b4, b5, b6, b7, b8],
            axis=-1
        )

        return output

# def inception_1d(x, is_train, depth, norm_function, active_function, name=None):
#     """Inception 1D module implementation.
#     :param x: input to current module (4D tensor with channels-last)
#     :param is_train: it is intented to be a boolean placeholder for controling the BatchNormalization behavior (0D tensor)
#     :param depth: linearly controls the depth of the network (int)
#     :param norm_function: normalization class (same format as the BatchNorm class above)
#     :param active_function: tensorflow activation function (e.g. tf.nn.relu)
#     :param name: of the variable scpoe (str)
#     """

#     init = tf.keras.initializers.GlorotUniform()
#     x_norm = norm_function(name="norm_input")(x, training=is_train)

#     # Branch 1: 64 x copy 1x1
#     branch_conv_1_1 = tf.keras.layers.Conv1D(filters=16 * depth, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#     branch_conv_1_1 = norm_function(name="norm_conv_1_1")(branch_conv_1_1, training=is_train)
#     branch_conv_1_1 = active_function(branch_conv_1_1, name="activation_1_1")
    
#     # Branch 2: 128 x conv 3x3
#     branch_conv_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#     branch_conv_3_3 = norm_function(name="norm_conv_3_3_1")(branch_conv_3_3, training=is_train)
#     branch_conv_3_3 = active_function(branch_conv_3_3, "activation_3_3_1")

#     branch_conv_3_3 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=3, padding="same", kernel_initializer=init)(branch_conv_3_3)
#     branch_conv_3_3 = norm_function(name="norm_conv_3_3_2")(branch_conv_3_3, training=is_train)
#     branch_conv_3_3 = active_function(branch_conv_3_3, "activation_3_3_2")

#     # Branch 3: 128 x conv 5x5
#     branch_conv_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#     branch_conv_5_5 = norm_function(name="norm_conv_5_5_1")(branch_conv_5_5, training=is_train)
#     branch_conv_5_5 = active_function(branch_conv_5_5, "activation_5_5_1")

#     branch_conv_5_5 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=5, padding="same", kernel_initializer=init)(branch_conv_5_5)
#     branch_conv_5_5 = norm_function(name="norm_conv_5_5_2")(branch_conv_5_5, training=is_train)
#     branch_conv_5_5 = active_function(branch_conv_5_5, "activation_5_5_2")

#     # Branch 4: 128 x conv 7x7
#     branch_conv_7_7 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(x_norm)
#     branch_conv_7_7 = norm_function(name="norm_conv_7_7_1")(branch_conv_7_7, training=is_train)
#     branch_conv_7_7 = active_function(branch_conv_7_7, "activation_7_7_1")

#     branch_conv_7_7 = tf.keras.layers.Conv1D(filters=32 * depth, kernel_size=5, padding="same", kernel_initializer=init)(branch_conv_7_7)
#     branch_conv_7_7 = norm_function(name="norm_conv_7_7_2")(branch_conv_7_7, training=is_train)
#     branch_conv_7_7 = active_function(branch_conv_7_7, "activation_7_7_2")

    
#     # Branch 5: 16 x (max_pool 3x3 + conv 1x1)
#     branch_maxpool_3_3 = tf.keras.layers.MaxPooling1D(pool_size=3, strides=1, padding="same", name="maxpool_3")(x_norm)
#     branch_maxpool_3_3 = norm_function(name="norm_maxpool_3_3")(branch_maxpool_3_3, training=is_train)
#     branch_maxpool_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_maxpool_3_3)
#     # Branch 6: 16 x (max_pool 5x5 + conv 1x1)
#     branch_maxpool_5_5 = tf.keras.layers.MaxPooling1D(pool_size=5, strides=1, padding="same", name="maxpool_5")(x_norm)
#     branch_maxpool_5_5 = norm_function(name="norm_maxpool_5_5")(branch_maxpool_5_5, training=is_train)
#     branch_maxpool_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_maxpool_5_5)
#     # Branch 7: 16 x (max_pool 3x3 + conv 1x1)
#     branch_avgpool_3_3 = tf.keras.layers.AveragePooling1D(pool_size=3, strides=1, padding="same", name="avgpool_3")(x_norm)
#     branch_avgpool_3_3 = norm_function(name="norm_avgpool_3_3")(branch_avgpool_3_3, training=is_train)
#     branch_avgpool_3_3 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_avgpool_3_3)
#     # Branch 8: 16 x (max_pool 5x5 + conv 1x1)
#     branch_avgpool_5_5 = tf.keras.layers.AveragePooling1D(pool_size=5, strides=1, padding="same", name="avgpool_5")(x_norm)
#     branch_avgpool_5_5 = norm_function(name="norm_avgpool_5_5")(branch_avgpool_5_5, training=is_train)
#     branch_avgpool_5_5 = tf.keras.layers.Conv1D(filters=16, kernel_size=1, padding="same", kernel_initializer=init)(branch_avgpool_5_5)
    
    
#     # Concatenate
#     output = tf.concat([branch_conv_1_1, branch_conv_3_3, branch_conv_5_5, branch_conv_7_7, branch_maxpool_3_3,
#                         branch_maxpool_5_5, branch_avgpool_3_3, branch_avgpool_5_5], axis=-1)
#     return output

## Load and prepare Data

In [ ]:
# Synthetic and provided noise addition
# filepaths_noise = glob.glob(os.path.join(get_train_audio_path(), "_background_noise_", "*.wav"))
filepaths_noise = glob.glob(os.path.join(get_train_audio_path(), "_background_noise_", "*.wav"))

noise = np.concatenate(list(map(lambda x: read_wav(x, False)[0], filepaths_noise)))
noise = np.concatenate([noise, noise[::-1]])
synthetic_noise = np.concatenate([white_noise(N=16000*30, state=np.random.RandomState(655321)),
                                  blue_noise(N=16000*30, state=np.random.RandomState(655321)),
                                  pink_noise(N=16000*30, state=np.random.RandomState(655321)),
                                  brown_noise(N=16000*30, state=np.random.RandomState(655321)),
                                  violet_noise(N=16000*30, state=np.random.RandomState(655321)),
                                  np.zeros(16000*60)])
synthetic_noise //= np.max(np.abs(synthetic_noise))
synthetic_noise = np.concatenate([synthetic_noise, (synthetic_noise+synthetic_noise[::-1])/2])
all_noise = np.concatenate([noise, synthetic_noise])

In [ ]:
np.random.seed(655321)
random.seed(655321)

path = get_silence_path()

if not os.path.exists(path):
    os.makedirs(path) # It fails in kaggle kernel due to the read-only filesystem

for noise_clip_no in tqdm(range(8000)):
    if noise_clip_no <= 4000:
        idx = np.random.randint(0, len(noise) - 16000)
        clip = noise[idx: (idx + 16000)]
    else:
        idx = np.random.randint(0, len(synthetic_noise) - 16000)
        clip = synthetic_noise[idx: (idx + 16000)]
    wavfile.write(os.path.join(path, "{0:04d}.wav".format(noise_clip_no)), 16000,
                  ((32767 * clip / np.max(np.abs(clip))).astype(np.int16)))

In [ ]:
filepaths = glob.glob(os.path.join(get_train_audio_path(), "**/*.wav"), recursive=True)
filepaths += glob.glob(os.path.join(get_silence_path(), "**/*.wav"), recursive=True)
filepaths = list(filter(lambda fp: "_background_noise_" not in fp, filepaths))
validation_list = open(os.path.join(get_train_path(), "validation_list.txt")).readlines()
test_list = open(os.path.join(get_train_path(), "testing_list.txt")).readlines()
validation_list = list(map(lambda fn: os.path.join(get_train_audio_path(), fn.strip()), validation_list))
testing_list = list(map(lambda fn: os.path.join(get_train_audio_path(), fn.strip()), test_list))
training_list = np.setdiff1d(filepaths, validation_list + testing_list).tolist()

In [ ]:
filepaths = glob.glob(os.path.join(get_train_audio_path(), "**/*.wav"), recursive=True)
filepaths += glob.glob(os.path.join(get_silence_path(), "**/*.wav"), recursive=True)
filepaths = list(filter(lambda fp: "_background_noise_" not in fp, filepaths))
validation_list = open(os.path.join(get_train_path(), "validation_list.txt")).readlines()
test_list = open(os.path.join(get_train_path(), "testing_list.txt")).readlines()
validation_list = list(map(lambda fn: os.path.join(get_train_audio_path(), fn.strip()), validation_list))
testing_list = list(map(lambda fn: os.path.join(get_train_audio_path(), fn.strip()), test_list))
training_list = np.setdiff1d(filepaths, validation_list + testing_list).tolist()

In [ ]:
random.seed(655321)
random.shuffle(filepaths)
random.shuffle(validation_list)
random.shuffle(test_list)
random.shuffle(training_list)

In [ ]:
#### 코드가 동일한데 assert 값이 달라 추후 확인 예정
# Quick Unit-Tests
# Test number of files and their consistencies
assert all(map(lambda fp: os.path.splitext(fp)[1] == ".wav", filepaths))
assert len(filepaths) == 64727 - 6 + 8000
# assert len(training_list) == len(filepaths) - 6798 - 6835
# (72721, 59088)
assert len(validation_list) == 6798
assert len(testing_list) == 6835

# Test file existence
assert all(map(lambda fn: os.path.exists(os.path.join(fn)), validation_list))
assert all(map(lambda fn: os.path.exists(os.path.join(fn)), testing_list))
assert all(map(lambda fn: os.path.exists(os.path.join(fn)), training_list))
# assert set(validation_list + testing_list + training_list) == set(filepaths)
# 길이로 비교 시 (86354, 72721)

# Test non-overlap among sets
assert len(np.intersect1d(validation_list, testing_list)) == 0
assert len(np.intersect1d(training_list, testing_list)) == 0
assert len(np.intersect1d(training_list, validation_list)) == 0

In [ ]:
# Classes processing
cardinal_calsses = list(set(map(lambda fp:os.path.split(os.path.split(fp)[0])[1], filepaths)))
le_classes = LabelEncoder().fit(cardinal_calsses)
Counter(map(lambda fp: os.path.split(os.path.split(fp)[0]), filepaths));

In [ ]:
# Quick Unit-Tests
# Test data preparation
_gen_test = get_batcher(filepaths, 1000)
batch_a_wav, batch_a_target = next(_gen_test)
batch_b_wav, batch_b_target = next(_gen_test)
_gen_test_le = get_batcher(filepaths, 1000, label_encoder=le_classes)
batch_le_wav, batch_le_target = next(_gen_test_le)

# Test batch matrix shape coherences
assert batch_a_wav.shape == (1000, 16000, 1)
assert batch_le_wav.shape == (1000, 16000, 1)
assert batch_a_wav.shape == batch_b_wav.shape == batch_le_wav.shape

# Test batch reproductibility
assert np.sum(np.abs(batch_a_wav - batch_b_wav)) != 0
assert len(batch_a_target) == len(batch_b_target) == len(batch_le_target)
assert any(batch_a_target != batch_b_target)

# Test classes label encoder
assert all(batch_le_target == np.expand_dims(le_classes.transform(np.squeeze(batch_a_target)), 1))

## Architecture design

Here it comes, WavCeption design

In [ ]:
class Architecture(tf.keras.Model):

    def __init__(self,
                 class_cardinality,
                 seq_len=16000,
                 name="architecture"):

        super().__init__(name=name)

        self.seq_len = seq_len
        self.class_cardinality = class_cardinality

        self.blocks = [

            Inception1D(depth=1, name="Inception_1_1"),
            Inception1D(depth=1, name="Inception_1_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_1"),

            Inception1D(depth=1, name="Inception_2_1"),
            Inception1D(depth=1, name="Inception_2_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_2"),

            Inception1D(depth=2, name="Inception_3_1"),
            Inception1D(depth=2, name="Inception_3_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_3"),

            Inception1D(depth=2, name="Inception_4_1"),
            Inception1D(depth=2, name="Inception_4_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_4"),

            Inception1D(depth=3, name="Inception_5_1"),
            Inception1D(depth=3, name="Inception_5_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_5"),

            Inception1D(depth=3, name="Inception_6_1"),
            Inception1D(depth=3, name="Inception_6_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_6"),

            Inception1D(depth=4, name="Inception_7_1"),
            Inception1D(depth=4, name="Inception_7_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_7"),

            Inception1D(depth=4, name="Inception_8_1"),
            Inception1D(depth=4, name="Inception_8_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_8"),

            Inception1D(depth=4, name="Inception_9_1"),
            Inception1D(depth=4, name="Inception_9_2"),
            tf.keras.layers.MaxPooling1D(2,2,name="maxpool_9"),
        ]

        self.flatten = tf.keras.layers.Flatten()

        self.bn_dense_1 = BatchNorm(name="bn_dense_1")

        self.dense_1 = tf.keras.layers.Dense(
            128,
            activation="relu"
        )

        self.bn_dense_2 = BatchNorm(name="bn_dense_2")

        self.output_dense = tf.keras.layers.Dense(
            class_cardinality
        )

    def call(self, x, training=False):

        for layer in self.blocks:

            if isinstance(layer, Inception1D):
                x = layer(x, training=training)
            else:
                x = layer(x)

        x = self.flatten(x)

        x = self.bn_dense_1(x, training=training)

        x = self.dense_1(x)

        x = self.bn_dense_2(x, training=training)

        x = self.output_dense(x)

        return x

# class Architecture(tf.keras.Model):
#     def __init__(self, class_cardinality, seq_len=16000, name="architecture"):
#         super().__init__(name=name)

#         self.seq_len = seq_len
#         self.class_cardinality = class_cardinality

#         self.flatten = tf.keras.layers.Flatten(name="flatten")

#         self.bn_dense_1 = BatchNorm(name="bn_dense_1")
#         self.dense_1 = tf.keras.layers.Dense(
#             128,
#             activation="relu",
#             kernel_initializer="glorot_uniform",
#             name="dense_1"
#         )

#         self.bn_dense_2 = BatchNorm(name="bn_dense_2")
#         self.output_dense = tf.keras.layers.Dense(
#             class_cardinality,
#             activation=None,
#             kernel_initializer="glorot_uniform",
#             name="output"
#         )

#     def call(self, x, training=False):

#         # Block 1
#         x = inception_1d(x=x, is_train=training, depth=1, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_1_1")
#         x = inception_1d(x=x, is_train=training, depth=1, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_1_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_1")(x)

#         # Block 2
#         x = inception_1d(x=x, is_train=training, depth=1, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_2_1")
#         x = inception_1d(x=x, is_train=training, depth=1, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_2_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_2")(x)

#         # Block 3
#         x = inception_1d(x=x, is_train=training, depth=2, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_3_1")
#         x = inception_1d(x=x, is_train=training, depth=2, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_3_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_3")(x)

#         # Block 4
#         x = inception_1d(x=x, is_train=training, depth=2, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_4_1")
#         x = inception_1d(x=x, is_train=training, depth=2, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_4_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_4")(x)

#         # Block 5
#         x = inception_1d(x=x, is_train=training, depth=3, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_5_1")
#         x = inception_1d(x=x, is_train=training, depth=3, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_5_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_5")(x)

#         # Block 6
#         x = inception_1d(x=x, is_train=training, depth=3, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_6_1")
#         x = inception_1d(x=x, is_train=training, depth=3, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_6_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_6")(x)

#         # Block 7
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_7_1")
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_7_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_7")(x)

#         # Block 8
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_8_1")
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_8_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_8")(x)

#         # Block 9
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_9_1")
#         x = inception_1d(x=x, is_train=training, depth=4, norm_function=BatchNorm, active_function=tf.nn.relu, name="Inception_9_2")
#         x = tf.keras.layers.MaxPooling1D(pool_size=2, strides=2, name="maxpool_9")(x)

#         # Dense
#         x = self.flatten(x)

#         x = self.bn_dense_1(x, training=training)
#         x = self.dense_1(x)

#         x = self.bn_dense_2(x, training=training)
#         x = self.output_dense(x)

#         return x

## Run model

You should use a GPU to run the model if you don't want it to take forever... In addition, you should decide when to stop the network to make the prediction. It took me 12h in a Titan X Pascal.

In [ ]:
net = Architecture(class_cardinality=len(cardinal_calsses), name="wavception")

In [ ]:
# sess = start_tensorflow_session(device="0")
# sw = get_summary_writer("~/.logs_ensorboard/", "wavception", "V1") # Adjust your tensorboard logs path here
# c = 0

start_tensorflow_session(device="0")

sw = get_summary_writer(
    "./logs_tensorboard/",
    "wavception",
    "V1"
)

c = 0

In [ ]:
import random

random.randint


import numpy

numpy.array

In [ ]:
# # TensorBoard graph tracing

# dummy_x = tf.random.normal((1, 16000, 1))

# tf.summary.trace_on(
#     graph=True,
#     profiler=False
# )

# _ = net(dummy_x, training=False)

# with sw.as_default():

#     tf.summary.trace_export(
#         name="wavception_trace",
#         step=0
#     )

# sw.flush()

In [ ]:
# # TensorBoard graph tracing
# dummy_x = tf.random.normal((1, 16000, 1))

# # tf.summary.trace_on(graph=True, profiler=True)
# tf.summary.trace_on(graph=False, profiler=False)

# _ = net(dummy_x, training=False)

# with sw.as_default():
#     tf.summary.trace_export(
#         name="wavception_trace",
#         step=0,
#         profiler_outdir="./logs_tensorboard/profile"
#     )
#     sw.flush()
# # sess.run(tf.global_variables_initializer())

In [ ]:
np.random.seed(655321)
random.seed(655321)

In [ ]:
optimizer = tf.keras.optimizers.Adam()
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

train_writer = tf.summary.create_file_writer("./logs/train")
val_writer = tf.summary.create_file_writer("./logs/val")

global_step = 0
random.shuffle(training_list)
# training_list= training_list[:100]
random.shuffle(validation_list)
# validation_list= validation_list[:100]
for epoch in range(50000):
# for epoch in range(5):
    batcher = get_batcher(training_list, 16, le_classes)
    
    for i, (batch_x, batch_y) in enumerate(batcher):
        batch_x = tf.convert_to_tensor(batch_x, tf.float32)
        batch_y = tf.convert_to_tensor(batch_y, tf.float32)

        with tf.GradientTape() as tape:
            logits = net(batch_x, training=True)
            loss = loss_fn(batch_y, logits)
        
        gradients = tape.gradient(loss, net.trainable_variables)

        optimizer.apply_gradients(zip(gradients, net.trainable_variables))

        train_acc_metric.update_state(batch_y, logits)

        acc = train_acc_metric.result().numpy()

        print("[{0:04d}|{1:04d}] Accuracy train: {2:.2f}%".format(epoch, i, acc*100))

        with train_writer.as_default():
            tf.summary.scalar("loss", loss, step=global_step)
            tf.summary.scalar("accuracy", acc, step=global_step)

        if global_step % 1000 == 0:
            accuracies_dev = []
            losses_dev = []

            batcher_val = get_batcher(validation_list, 16, le_classes)

            for val_x, val_y in tqdm(batcher_val):
                val_x = tf.convert_to_tensor(val_x, dtype=tf.float32)
                val_y = tf.convert_to_tensor(val_y, dtype=tf.float32)
                val_logits = net(val_x, training=False)
                val_loss = loss_fn(val_y, val_logits)

                val_acc_metric.update_state(val_y, val_logits)
                val_acc = val_acc_metric.result().numpy()
                accuracies_dev.append(val_acc)
                losses_dev.append(val_loss.numpy())
            
            mean_acc = np.mean(accuracies_dev)
            mean_loss = np.mean(losses_dev)

            with val_writer.as_default():
                tf.summary.scalar("val_loss", mean_loss, step=global_step)
                tf.summary.scalar("val_accuracy", mean_acc, step=global_step)

            print(f"[Validation] loss={mean_loss:.4f}, acc={mean_acc:.4f}")

            val_acc_metric.reset_state()
        
        global_step += 1
    
    train_acc_metric.reset_state()
    

https://www.kaggle.com/code/ivallesp/wavception-v1-a-1-d-inception-approach-lb-0-76